# CLADES back-test: does the Beneish M-score flag known Indian cases early?

This notebook is **offline research**. It reads `notebooks/cases.csv`, which you fill by hand from original annual reports
(start from `cases_template.csv`), and writes `config/validation.json`, the static file the app's Validation page shows.

Ground rules, enforced by `scripts/backtest.py`:

- **No invented figures.** A blank cell stays blank; that company-year gets no M-score and is listed as skipped.
- **No look-ahead.** A case counts as positive only in the years strictly *before* its event year, using accounts as first filed.
- **Out-of-period test.** A tuned threshold is chosen on the earlier period only and applied unchanged to the later period.
- **Both error types.** Hit rate is always shown next to the false-positive rate on the clean control group.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
import backtest

backtest.selftest()            # the Python formulas must match the app's hand-calculated example
CASES = ROOT / "notebooks" / "cases.csv"
SPLIT_FY = 2014                # thresholds are fixed on FY <= SPLIT_FY and tested on later years
LOOKBACK = 3                   # years before the event that count as "should have been flagged"
SOURCE = "Annual reports, figures entered by hand"

In [ ]:
rows = backtest.load_cases(CASES) if CASES.exists() else []
if not rows:
    print("No data yet. Copy cases_template.csv to cases.csv and fill it from original annual reports.")
    print("Nothing below will produce numbers until then, by design.")
else:
    scored, skipped = backtest.score(rows)
    print(f"{len(rows)} company-years entered, {len(scored)} with a complete M-score, {len(skipped)} skipped")
    for line in skipped:
        print("  skipped:", line)

## M-score by company-year

`label` is `True` for a case inside the look-back window, `False` for a control, and `None` (excluded) for case years outside the window.

In [ ]:
if rows:
    print(f"{'company':32} {'FY':>5} {'group':8} {'M':>7}  label")
    for s in sorted(scored, key=lambda s: (s['company'], s['fy'])):
        flag = 'FLAG' if s['m'] >= backtest.PUBLISHED_THRESHOLD else ''
        print(f"{s['company'][:32]:32} {s['fy']:>5} {s['group']:8} {s['m']:>7.2f}  {backtest.label(s, LOOKBACK)!s:5} {flag}")

## Results

Read the **later-period** rows. The earlier-period rows are in-sample for the tuned threshold.

In [ ]:
if rows:
    result = backtest.evaluate(scored, skipped, SPLIT_FY, LOOKBACK, SOURCE)
    print("status:", result["status"], "-", result.get("note", ""))
    f = lambda v: "n/a" if v is None else f"{v:.2f}"
    for s in result.get("sets", []):
        print(f"{s['name']}\n   n={s['n']}  hit rate={f(s['hitRate'])}  false-positive rate={f(s['falsePositiveRate'])}  precision={f(s['precision'])}  AUC={f(s['auc'])}  TP/FP/FN/TN={s['tp']}/{s['fp']}/{s['fn']}/{s['tn']}")
    print()
    for e in result.get("earlyWarning", []):
        print(f"{e['company']}: event FY{e['eventFy']}, first flagged {('FY' + str(e['firstFlagFy'])) if e['firstFlagFy'] else 'never'}, years early: {e['yearsEarly']}")

## Base rates: what a flag is actually worth

The sample has roughly as many cases as controls, so precision above is far higher than it would be in the market, where
real problems are rare. The table below re-computes precision for realistic prevalence. Expect it to be low: with a 2% base rate,
even a 70% hit rate and a 15% false-positive rate means fewer than one flag in ten is a real problem.

In [ ]:
if rows and result["status"] == "ok":
    for b in result["baseRates"]:
        p = "n/a" if b["precision"] is None else f"{b['precision']:.0%}"
        print(f"If {b['prevalence']:.0%} of companies truly have a problem, a flag is right about {p} of the time")
    print()
    for c in result["caveats"]:
        print("-", c)

## If the result looks too good, look for the leak first

Before believing a strong result, check each of these:

1. **Restated or vendor data.** Were any figures taken from a data provider or a later report? Those include restatements made *after* the event.
2. **Event year inside the window.** `label` must be `None`, not `True`, for the event year and later.
3. **Control group chosen after the fact.** Were controls picked after seeing their scores, or only from companies known to be fine today (survivors)?
4. **Threshold tuned on the test period.** The tuned threshold must come from `FY <= SPLIT_FY` only. Changing `SPLIT_FY` until the numbers look good is tuning on the test set.
5. **Tiny denominators.** A 100% hit rate on three positives is three companies, not a property of the model.
6. **Sector mix.** If cases are airlines and jewellers and controls are IT services, the score may be separating sectors, not behaviour.

In [ ]:
if rows:
    out = ROOT / "config" / "validation.json"
    out.write_text(__import__("json").dumps(result, indent=2) + "\n", encoding="utf-8")
    print("wrote", out, "- commit it to publish the Validation page")